<a href="https://colab.research.google.com/github/john2004vincent/Data-Science-Coursework-1/blob/main/Fish_Dataset_Feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Kaggle package

In [ ]:
! pip install kaggle

Upload your Kaggle API json file

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"john04vincent","key":"4b124eec5270d27fc442eb81898edf68"}'}

Create a Kaggle Directory and provide permissions

In [ ]:
! mkdir ~/.kaggle #create a kaggle directory
! cp kaggle.json ~/.kaggle/ #copy the api token to the directory
! chmod 600 ~/.kaggle/kaggle.json #change the access type of the kaggle directory

Download the Kaggle dataseet from the web

In [ ]:
! kaggle datasets download -d crowww/a-large-scale-fish-dataset

Dataset URL: https://www.kaggle.com/datasets/crowww/a-large-scale-fish-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 3.23G/3.24G [00:35<00:00, 105MB/s]
100% 3.24G/3.24G [00:35<00:00, 98.8MB/s]


Import Necesary libraries for zip extraction

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
from skimage import measure
import cv2
from os.path import exists, join, basename, splitext


Unzip the kaggle file to the content forlder

In [ ]:
zip_file_path = '/content/a-large-scale-fish-dataset.zip'
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall('/content/')

In [ ]:
fishes_path = '/content/Fish_Dataset/Fish_Dataset' #this string will be used all throughout the process
print(os.listdir(fishes_path))


['Hourse Mackerel', 'Sea Bass', 'Trout', 'Shrimp', 'license.txt', 'Striped Red Mullet', 'Segmentation_example_script.m', 'README.txt', 'Red Mullet', 'Red Sea Bream', 'Gilt-Head Bream', 'Black Sea Sprat']


Create a list of paths to be accessed. Remove unecessary paths such as txt and script files. Both the original picture and the GT picture are necessary.

In [ ]:
paths = {} #a hashmap is used for an easier and more organized access
for i in os.listdir(fishes_path):
  if '.' in i: #deny any type of file that is not a folder
    pass
  else:
    p = fishes_path + '/' + i
    if 'GT' in os.listdir(p)[0]: #let the GT images serve as values and its normal version serve as keys
      paths[os.listdir(p)[1]] = os.listdir(p)[0]
    else:
      paths[os.listdir(p)[0]] = os.listdir(p)[1]
paths

{'Red Sea Bream': 'Red Sea Bream GT',
 'Trout': 'Trout GT',
 'Red Mullet': 'Red Mullet GT',
 'Black Sea Sprat': 'Black Sea Sprat GT',
 'Shrimp': 'Shrimp GT',
 'Sea Bass': 'Sea Bass GT',
 'Striped Red Mullet': 'Striped Red Mullet GT',
 'Gilt-Head Bream': 'Gilt-Head Bream GT',
 'Hourse Mackerel': 'Hourse Mackerel GT'}

Function to extract features. Features set here will also be used to derive more useful features. The function returns a dictionary of the set features. Running this part also resets the dataframe.

In [ ]:
properties = ['label',
              'area',
              'equivalent_diameter',
              'mean_intensity',
              'solidity',
              'axis_major_length', 'axis_minor_length']
#set the columns to be used by the dataframe

df = pd.DataFrame(columns = properties) #reset dataframe
def feature_extraction(label_image, image_for_label): #measure properties using the sci kit image measure package
  props = measure.regionprops_table(label_image,image_for_label,
                                  properties = ['label',
                                                'area',
                                                'equivalent_diameter',
                                                'mean_intensity',
                                                'solidity',
                                                'axis_major_length', 'axis_minor_length'])

  return (props)

df_names = pd.DataFrame(columns = ['name']) #a completely different dataframe that will hold the names of the fish; will be merged with the first dataframe

A nested loop goes through each file in the folder. The images used are the 10th to 15th images per fish. All 9 fish are used.

In [ ]:
row_count = 0
for i in paths.keys(): # loop through the names of the folder
  label_image_file = fishes_path + '/' + i + '/' + paths[i] #ex: '/content/Fish_Dataset/Fish_Dataset/Black Sea Sprat/Black Sea Sprat GT'
  image_for_label_file = fishes_path + '/' + i + '/' + i #ex: '/content/Fish_Dataset/Fish_Dataset/Black Sea Sprat/Black Sea Sprat'
  for j in range(10,15): # loop through all images in the folder
    imshowpic = os.listdir(label_image_file)[j] #ex: '00983.png'; to be appended with the original name; stays same for both folders
    final_label = label_image_file + '/' + imshowpic
    final_for_label = image_for_label_file + '/' + imshowpic
    df.loc[row_count] = feature_extraction((cv2.imread(final_label)),
                                           (cv2.imread(final_for_label))) #apply feature extraction and add it to the dataframe
    df_names.loc[row_count] = i #take the name of the fish

    print(df.loc[[row_count]])
    print(df_names.loc[[row_count]])
    row_count += 1


   label      area  equivalent_diameter      mean_intensity  \
0  [255]  [174396]  [69.31799074849721]  [126.311010573637]   

               solidity    axis_major_length     axis_minor_length  
0  [0.8934312851566102]  [508.4510721291587]  [3.6514837167010037]  
            name
0  Red Sea Bream
   label      area  equivalent_diameter       mean_intensity  \
1  [255]  [166623]  [68.27244282386765]  [129.8327181721611]   

              solidity    axis_major_length     axis_minor_length  
1  [0.918442941478015]  [395.2364829287016]  [3.6514837167000698]  
            name
1  Red Sea Bream
   label      area  equivalent_diameter        mean_intensity  \
2  [255]  [225834]  [75.55506232974075]  [139.22421778828698]   

              solidity   axis_major_length    axis_minor_length  
2  [0.881619937694704]  [597.404585868101]  [3.651483716705051]  
            name
2  Red Sea Bream
   label      area  equivalent_diameter        mean_intensity  \
3  [255]  [141540]  [64.65868327967931] 

Deriving even more features and cleaning of the final dataframe. Some functions are used due to issues with numpy functions being unable to fully interact with dataframes.

In [ ]:
full_df = pd.concat([df, df_names], axis=1) #merge dataframes to give rows a fish name
full_df['area_sqs_microns'] = full_df['area'] * (0.6 ** 2)
full_df['equivalent_dimater_microns'] = full_df['equivalent_diameter'] * 0.6
full_df['elongation'] = full_df['axis_minor_length'] / full_df['axis_major_length']
def calculate_perimeter(row):
    a = row['axis_major_length']
    b = row['axis_minor_length']
    perimeter = np.pi * (3 * (a + b) - np.sqrt((3*a + b) * (a + 3*b)))
    return perimeter

full_df['perimeter'] = full_df.apply(calculate_perimeter, axis=1)
#https://www.mathsisfun.com/geometry/ellipse-perimeter.html

def calculate_eccentricity(row):
    a = row['axis_major_length']
    b = row['axis_minor_length']
    eccentricity = np.sqrt(1 - (b / a)**2)
    return eccentricity

full_df['eccentricity'] = full_df.apply(calculate_eccentricity, axis=1)
#https://byjus.com/maths/eccentricity/

full_df['circularity'] = 4 * np.pi * full_df['area'] / (full_df['perimeter'] ** 2)
#https://sciencing.com/calculate-circularity-5138742.html

full_df = full_df.drop('label', axis=1)
full_df

,area,equivalent_diameter,mean_intensity,solidity,axis_major_length,axis_minor_length,name,area_sqs_microns,equivalent_dimater_microns,elongation,perimeter,eccentricity,circularity
0,[174396],[69.31799074849721],[126.311010573637],[0.8934312851566102],[508.4510721291587],[3.6514837167010037],Red Sea Bream,[62782.56],[41.590794449098325],[0.007181583276853509],[2026.7782337228064],[0.9999742120983108],[0.5334993816745869]
1,[166623],[68.27244282386765],[129.8327181721611],[0.918442941478015],[395.2364829287016],[3.6514837167000698],Red Sea Bream,[59984.28],[40.96346569432059],[0.009238731428947506],[1575.8368795703204],[0.9999573220100865],[0.843183876544379]
2,[225834],[75.55506232974075],[139.22421778828698],[0.881619937694704],[597.404585868101],[3.651483716705051],Red Sea Bream,[81300.23999999999],[45.33303739784445],[0.006112245876718546],[2381.095393966447],[0.9999813200507011],[0.5005478508837591]
3,[141540],[64.65868327967931],[126.44652395082662],[0.8913491148854169],[447.57372841297],[3.651483716701315],Red Sea Bream,[50954.4],[38.79520996780759],[0.008158396002484181],[1784.2974688504903],[0.9999667197335452],[0.5586689524386865]
4,[216615],[74.51264298886205],[125.99433095584331],[0.8865600903688424],[567.81365760295],[3.6514837167056737],Red Sea Bream,[77981.4],[44.70758579331723],[0.0064307782453147935],[2263.228996289323],[0.9999793223317969],[0.5314242812265986]
5,[130350],[62.90774790413641],[128.5196854622171],[0.896671275563902],[478.6929553420108],[3.6514837167000698],Trout,[46926.0],[37.74464874248184],[0.007628028939952128],[1908.2480447711312],[0.99997090616402],[0.44983293271404223]
6,[244491],[77.58088893604521],[119.94274635876167],[0.9385162835690267],[563.2601893283265],[3.651483716698824],Trout,[88016.76],[46.54853336162713],[0.006482765488988536],[2245.0917012084487],[0.999978986655027],[0.6095433138416954]
7,[227034],[75.68865006571953],[124.3637340662632],[0.9412920719420881],[587.5764986810743],[3.651483716701315],Trout,[81732.23999999999],[45.41319003943172],[0.00621448224171279],[2341.9481585260874],[0.9999806899187941],[0.5201710887160457]
8,[151575],[66.15200090537809],[96.5490351311232],[0.867411756626837],[534.9058711939875],[3.651483716704117],Trout,[54567.0],[39.69120054322685],[0.006826404257918268],[2132.151618276535],[0.9999766998310048],[0.4189876985191968]
9,[191970],[71.57229020337213],[122.00712611345523],[0.928750780127433],[538.7291235171936],[3.6514837167010037],Trout,[69109.2],[42.94337412202328],[0.006777958638771209],[2147.3802480143618],[0.9999770293745207],[0.523148905151362]


Download the final dataframe as a csv file.

In [ ]:
full_df.to_csv('Processed_fresh_fish.csv', index=True)